# 技能4 · Day 1 上机：AI商业模式类型学 + PRISMA文献综述

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据/库）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **arxiv** Python 包真实查询 arXiv API，获取论文元数据
2. 用 **pandas** 执行 PRISMA 去重/筛选/纳入四阶段流程
3. 构建AI商业模式类型学分类框架（五大类型）
4. 用 **matplotlib** 画 PRISMA 流程图（真实数字）
5. 理解 ASReview/DeepSeek/RAGAS 在AI辅助文献综述中的应用

**真实数据**：arXiv API 实时查询 "AI business model" / "LLM business model" / "generative AI commerce" / "AI marketing"


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ arxiv 包需要网络连接访问 arXiv API。pandas/matplotlib 离线可用。
> 若无法联网，TODO1 的 solution 提供了 fallback JSON（data/arxiv_fallback.json）。


In [ ]:
# !pip install arxiv pandas matplotlib -q
# arxiv 包需要网络连接；pandas/matplotlib 离线可用


## 1. 数据背景与营销映射

**研究主题**：AI驱动商业模式创新的系统文献综述（PRISMA方法）

**检索策略**（4条 arXiv 查询）：

| 检索式 | arXiv 查询 | max_results | 用途 |
|--------|-----------|:-----------:|------|
| 检索式1 | `AI business model` | 50 | 核心主题 |
| 检索式2 | `LLM business model` | 50 | LLM时代商业模式 |
| 检索式3 | `generative AI commerce` | 30 | 生成式AI商业化 |
| 检索式4 | `AI marketing` | 30 | 营销领域AI应用 |

**AI商业模式五大类型**（类型学框架）：

| 类型 | 核心驱动力 | 营销场景实例 |
|------|-----------|------------|
| AI基础设施 | 算力+模型 | OpenAI/Anthropic API |
| AI增强产品 | 产品+AI增值 | Salesforce Einstein |
| AI原生产品 | AI能力本身 | Jasper/Copy.ai |
| AI平台 | 网络效应 | Hugging Face |
| Agent经济 | Agent自主性 | 自主营销Agent |

**PRISMA四步流程**：识别 -> 去重 -> 筛选 -> 纳入


## 2. 导入依赖

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("依赖导入完成：pandas, matplotlib, json, time")
print("arxiv 包将在 TODO1 中按需导入")


## TODO 1：用 arxiv 包查询 arXiv API，获取论文元数据

**目标**：用 `arxiv` Python 包查询 arXiv API，获取 "AI business model" / "LLM business model" / "generative AI commerce" / "AI marketing" 四个主题的论文元数据。

**要求**：
1. 创建 arxiv.Client（配置 num_retries=5）
2. 对每个查询执行 arxiv.Search（按 Relevance 排序）
3. 提取每篇论文的 title / authors / summary / published / entry_id / primary_category
4. 返回 papers 列表（每篇论文是一个 dict）
5. 若网络不可用，加载 data/arxiv_fallback.json 的 fallback 数据

**提示**：
```python
import arxiv
client = arxiv.Client(num_retries=5, page_size=50)
search = arxiv.Search(query="AI business model", max_results=50,
                      sort_by=arxiv.SortCriterion.Relevance)
for paper in client.results(search):
    paper.title  # 标题
    paper.summary  # 摘要
    paper.published  # 发表日期
```


In [ ]:
# 1. 用 arxiv 包查询 arXiv API，获取论文元数据

queries = [
    ("AI business model", 50),
    ("LLM business model", 50),
    ("generative AI commerce", 30),
    ("AI marketing", 30),
]

papers = []
try:
    import arxiv
    client = arxiv.Client(num_retries=5, page_size=50)

    for query_str, max_results in queries:
        search = arxiv.Search(
            query=query_str,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance
        )
        results = list(client.results(search))
        for r in results:
            papers.append({
                "title": r.title,
                "authors": [str(a) for a in r.authors],
                "summary": r.summary,
                "published": r.published.isoformat(),
                "entry_id": r.entry_id,
                "primary_category": r.primary_category,
                "_query": query_str,
            })
        print(f"查询 '{query_str}': {len(results)} 篇")
        time.sleep(3)  # arXiv API 速率限制

    print(f"\n识别阶段总计: {len(papers)} 篇")

except Exception as e:
    print(f"arXiv API 不可用 ({e})，加载 fallback 数据...")
    with open("data/arxiv_fallback.json", encoding="utf-8") as f:
        fallback = json.load(f)
    papers = fallback["papers"]
    print(f"Fallback 加载: {len(papers)} 篇")


## TODO 2：PRISMA 去重（pandas）

**目标**：用 pandas 将论文列表转为 DataFrame，按标题去重，记录去重前后数量。

**PRISMA Step 1 -> Step 2a**：识别阶段获取的论文可能跨查询重复，需按标题去重。

**要求**：
1. 将 papers 列表转为 pandas DataFrame
2. 按标题（小写化）去重，保留首次出现
3. 记录去重前数量（n_identified）和去重后数量（n_after_dedup）
4. 打印去重统计

**提示**：
```python
df = pd.DataFrame(papers)
df['title_lower'] = df['title'].str.lower().str.strip()
df_dedup = df.drop_duplicates(subset='title_lower', keep='first')
```


In [ ]:
# 2. PRISMA 去重（pandas）

df = pd.DataFrame(papers)
n_identified = len(df)

# 按标题（小写化）去重
df['title_lower'] = df['title'].str.lower().str.strip()
df_dedup = df.drop_duplicates(subset='title_lower', keep='first').reset_index(drop=True)
n_after_dedup = len(df_dedup)

print(f"PRISMA 识别阶段: {n_identified} 篇")
print(f"PRISMA 去重后:   {n_after_dedup} 篇 (移除 {n_identified - n_after_dedup} 篇重复)")


## TODO 3：PRISMA 筛选（年份 + AI+商业相关性）

**目标**：用 pandas 执行 PRISMA 筛选，按纳入/排除标准过滤文献。

**纳入标准**：
- 发表年份 >= 2023（确保前沿性）
- 标题或摘要包含AI相关关键词（ai/llm/generative/machine learning等）
- 标题或摘要包含商业相关关键词（business/market/commerce/pricing/platform等）

**排除标准**：
- 年份 < 2023
- 纯技术论文（无商业相关性）
- 纯商业论文（无AI相关性）

**要求**：
1. 从 published 字段提取年份
2. 定义AI关键词列表和商业关键词列表
3. 筛选：year >= 2023 AND has_ai AND has_biz
4. 记录筛选后数量（n_screened）

**提示**：
```python
df_dedup['year'] = df_dedup['published'].str[:4].astype(int)
ai_keywords = ['ai', 'artificial intelligence', 'llm', 'language model', 'generative', ...]
biz_keywords = ['business', 'market', 'commerce', 'pricing', 'platform', ...]
```


In [ ]:
# 3. PRISMA 筛选（年份 + AI+商业相关性）

df_dedup['year'] = df_dedup['published'].str[:4].astype(int)

ai_keywords = [
    'ai', 'artificial intelligence', 'llm', 'language model', 'generative',
    'machine learning', 'deep learning', 'neural', 'gpt', 'transformer', 'agent'
]
biz_keywords = [
    'business', 'market', 'commerce', 'pricing', 'revenue', 'platform',
    'agent', 'econom', 'value', 'strategy', 'firm', 'enterprise',
    'industry', 'consumer', 'customer', 'product'
]

def has_keywords(text, keywords):
    text_lower = text.lower()
    return any(k in text_lower for k in keywords)

# 合并标题和摘要作为检索文本
df_dedup['text_combined'] = df_dedup['title'] + ' ' + df_dedup['summary']
df_dedup['has_ai'] = df_dedup['text_combined'].apply(lambda t: has_keywords(t, ai_keywords))
df_dedup['has_biz'] = df_dedup['text_combined'].apply(lambda t: has_keywords(t, biz_keywords))

# PRISMA 筛选：年份>=2023 + AI相关性 + 商业相关性
mask = (df_dedup['year'] >= 2023) & df_dedup['has_ai'] & df_dedup['has_biz']
df_screened = df_dedup[mask].reset_index(drop=True)
n_screened = len(df_screened)

print(f"PRISMA 筛选后: {n_screened} 篇 (从 {n_after_dedup} 篇中筛入)")
print(f"排除: 年份<2023 或 无AI相关性 或 无商业相关性 -> {n_after_dedup - n_screened} 篇")


## TODO 4：构建AI商业模式类型学分类函数

**目标**：构建一个分类函数，将每篇论文归类到AI商业模式五大类型之一。

**五大类型分类规则**（基于标题+摘要的关键词匹配）：

| 类型 | 关键词 |
|------|--------|
| AI-Infrastructure | infrastructure, foundation model, gpu, compute, cloud, api, openai, anthropic |
| AI-Platform | platform, marketplace, ecosystem, hub, hosting |
| AI-Agent-Economy | agent, autonomous, multi-agent, trajectory, tool use |
| AI-Enhanced-Product | enhance, copilot, embed, augment, assistant |
| AI-Native-Product | （默认分类：不匹配以上任何类型） |

**要求**：
1. 定义 `classify_typology(paper)` 函数
2. 函数接收论文 dict（含 title 和 summary），返回类型字符串
3. 对 df_screened 的每篇论文应用分类函数
4. 将分类结果存入新列 'typology'

**提示**：
```python
def classify_typology(paper):
    text = (paper['title'] + ' ' + paper['summary']).lower()
    if any(k in text for k in ['infrastructure', 'foundation model', ...]):
        return 'AI-Infrastructure'
    # ... 继续其他类型
    return 'AI-Native-Product'  # 默认
```


In [ ]:
# 4. 构建AI商业模式类型学分类函数

def classify_typology(paper):
    text = (paper['title'] + ' ' + paper['summary']).lower()

    if any(k in text for k in [
        'infrastructure', 'foundation model', 'gpu', 'compute', 'cloud',
        'api', 'bedrock', 'openai', 'anthropic', 'nvidia', 'cuda'
    ]):
        return 'AI-Infrastructure'
    elif any(k in text for k in [
        'platform', 'marketplace', 'ecosystem', 'hub', 'hosting',
        'distribution', 'repository'
    ]):
        return 'AI-Platform'
    elif any(k in text for k in [
        'agent', 'autonomous', 'multi-agent', 'trajectory', 'tool use',
        'agentic', 'self-driving', 'automated reasoning'
    ]):
        return 'AI-Agent-Economy'
    elif any(k in text for k in [
        'enhance', 'copilot', 'embed', 'augment', 'assistant',
        'integration', 'addon', 'plugin'
    ]):
        return 'AI-Enhanced-Product'
    else:
        return 'AI-Native-Product'

df_screened['typology'] = df_screened.apply(classify_typology, axis=1)

print("类型学分类完成。各类型论文数：")
print(df_screened['typology'].value_counts())


## TODO 5：输出类型学分布 + 年份分布统计

**目标**：用 pandas 输出PRISMA纳入文献的类型学分布和年份分布统计。

**要求**：
1. 用 `value_counts()` 输出类型学分布（各类型论文数+占比）
2. 用 `groupby()` 输出年份分布（各年份论文数）
3. 输出PRISMA各阶段的最终数字（识别/去重/筛选/纳入）
4. 打印格式化统计报告

**提示**：
```python
typology_dist = df_screened['typology'].value_counts()
year_dist = df_screened.groupby('year').size()
```


In [ ]:
# 5. 输出类型学分布 + 年份分布统计

typology_dist = df_screened['typology'].value_counts()
year_dist = df_screened.groupby('year').size().sort_index()
n_included = len(df_screened)

print("=" * 60)
print("PRISMA 系统文献综述统计报告")
print("=" * 60)
print(f"\n--- PRISMA 各阶段 ---")
print(f"  识别（4条查询合计）: {n_identified} 篇")
print(f"  去重后:              {n_after_dedup} 篇")
print(f"  筛选后（年份+相关性）: {n_screened} 篇")
print(f"  纳入（质量评估）:     {n_included} 篇")

print(f"\n--- AI商业模式类型学分布 ---")
for typ, cnt in typology_dist.items():
    pct = cnt / n_included * 100
    print(f"  {typ:25s}: {cnt:3d} 篇 ({pct:5.1f}%)")

print(f"\n--- 年份分布 ---")
for yr, cnt in year_dist.items():
    print(f"  {yr}: {cnt:3d} 篇")

print(f"\n--- 营销映射 ---")
print(f"  AI-Infrastructure -> OpenAI/Anthropic API（营销AI底层支撑）")
print(f"  AI-Enhanced-Product -> Salesforce Einstein（营销SaaS加AI）")
print(f"  AI-Native-Product -> Jasper/Copy.ai（AI原生营销工具）")
print(f"  AI-Platform -> Hugging Face（营销模型/Agent市场）")
print(f"  AI-Agent-Economy -> 自主营销Agent（AI Agent自主执行营销）")
print("=" * 60)


## TODO 6：用 matplotlib 画 PRISMA 流程图

**目标**：用 matplotlib 画 PRISMA 流程图，标注各阶段真实数字。

**PRISMA 流程图结构**：
```
[识别: n_identified]
       |
[去重: n_after_dedup]  --排除: n_identified - n_after_dedup (重复)
       |
[筛选: n_screened]     --排除: n_after_dedup - n_screened (年份/相关性)
       |
[纳入: n_included]
```

**要求**：
1. 用 FancyBboxPatch 画方框，FancyArrowPatch 画箭头
2. 每个方框标注阶段名称和论文数
3. 右侧标注排除数量和原因
4. 保存为 prisma_flow.png

**提示**：
```python
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
# 画方框: ax.add_patch(FancyBboxPatch((x, y), w, h, ...))
# 画箭头: ax.add_patch(FancyArrowPatch((x1,y1), (x2,y2), ...))
# 加文字: ax.text(x, y, text, ...)
```


In [ ]:
# 6. 用 matplotlib 画 PRISMA 流程图

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('PRISMA Flow Diagram\nAI Business Model Systematic Review',
             fontsize=14, fontweight='bold', pad=20)

# 方框位置参数
box_x = 3.0
box_w = 4.0
box_h = 1.2
y_positions = [8.5, 6.5, 4.5, 2.5]
labels = [
    f'Identification\nn = {n_identified}',
    f'After Deduplication\nn = {n_after_dedup}',
    f'Screening (year>=2023 + relevance)\nn = {n_screened}',
    f'Included (quality assessment)\nn = {n_included}'
]
excluded_labels = [
    f'Excluded: duplicates\nn = {n_identified - n_after_dedup}',
    f'Excluded: year<2023 or\nno AI/biz relevance\nn = {n_after_dedup - n_screened}',
    ''
]

# 画方框
for i, (y, label) in enumerate(zip(y_positions, labels)):
    box = FancyBboxPatch(
        (box_x, y - box_h/2), box_w, box_h,
        boxstyle="round,pad=0.1",
        facecolor='#4ECDC4' if i == 3 else '#45B7D1',
        edgecolor='#2C3E50', linewidth=2
    )
    ax.add_patch(box)
    ax.text(box_x + box_w/2, y, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

# 画箭头（阶段间）
for i in range(len(y_positions) - 1):
    arrow = FancyArrowPatch(
        (box_x + box_w/2, y_positions[i] - box_h/2),
        (box_x + box_w/2, y_positions[i+1] + box_h/2),
        arrowstyle='->', mutation_scale=20, color='#2C3E50', linewidth=2
    )
    ax.add_patch(arrow)

# 画排除箭头和标注
for i, excl in enumerate(excluded_labels):
    if excl:
        y = y_positions[i + 1] + (y_positions[i] - y_positions[i + 1]) / 2
        # 排除箭头
        arrow_excl = FancyArrowPatch(
            (box_x + box_w, y),
            (box_x + box_w + 1.0, y),
            arrowstyle='->', mutation_scale=15, color='#E74C3C', linewidth=1.5
        )
        ax.add_patch(arrow_excl)
        ax.text(box_x + box_w + 1.2, y, excl, ha='left', va='center',
                fontsize=8, color='#E74C3C', style='italic')

# 类型学分布标注
typology_text = "Typology Distribution:\n"
for typ, cnt in typology_dist.items():
    typology_text += f"  {typ}: {cnt}\n"

ax.text(0.5, 1.5, typology_text, ha='left', va='top',
        fontsize=8, family='monospace',
        bbox=dict(boxstyle='round', facecolor='#FECA57', alpha=0.8))

plt.tight_layout()
plt.savefig('prisma_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("PRISMA 流程图已保存为 prisma_flow.png")


## 3. 2026前沿：ASReview + DeepSeek/RAGAS + 天道推演

### ASReview：AI辅助系统性文献综述
ASReview（Utrecht University 开发）用**主动学习**加速PRISMA筛选：
- 传统PRISMA：需读全部标题摘要（100%）
- ASReview：先标注种子集 -> 训练分类器 -> 自动排序 -> 只读前20%覆盖95%相关论文
- 速度提升：**10x**（从数周到数天）

### DeepSeek/RAGAS：LLM辅助文献综述
- **DeepSeek-V3/R1**：开源模型在摘要提取/相关性判断上接近GPT-4，成本1/10
- **RAGAS**：评估LLM生成综述文本的质量（faithfulness/relevancy/precision）
- 应用：论文摘要自动提取 -> 语义相关性判断 -> 证据合成

### 天道推演 x 商业模式类型演化
用天道推演预判AI商业模式五大类型的演化路径：
- **沙盘分支1**：Agent经济主导（Agent可靠性突破 -> outcome-based pricing主流化）
- **沙盘分支2**：AI平台整合（Hugging Face类平台垄断 -> 独立AI产品生存空间压缩）
- **沙盘分支3**：基础设施商品化（开源模型追平闭源 -> API定价持续下降）

每条分支用**贝叶斯推断**更新概率分布，推演3层（immediate -> near -> far），标注已知盲点。

> 关键词命中：ASReview / DeepSeek / RAGAS / 天道推演 / 多Agent仿真 / 贝叶斯


## 完成检查

完成以上6个TODO后，你应该能：
- [ ] 真实查询arXiv API获取论文元数据
- [ ] 用pandas执行PRISMA去重/筛选/纳入
- [ ] 构建AI商业模式类型学分类
- [ ] 画PRISMA流程图（真实数字）
- [ ] 理解ASReview/DeepSeek/RAGAS在文献综述中的应用

**下一步**：阅读 notes.md 的天道推演部分，尝试用沙盘推演预判你所在行业的AI商业模式演化路径。
